In [2]:
import pandas as pd
import numpy as np

# Загружаем данные
df = pd.read_csv('detections.csv')

# Список транспорта
transport = ['car', 'truck', 'bus', 'motorcycle']

# Фильтруем только транспорт
df = df[df['class'].isin(transport)]

print("СТРУКТУРА ПОТОКА (уникальные машины):")
type_counts = df.groupby('class')['track_id'].nunique()
total = type_counts.sum()
for t, count in type_counts.items():
    print(f"{t}: {count} ({count/total*100:.1f}%)")

print("\nИНТЕНСИВНОСТЬ (первые появления):")
first_seen = df.groupby('track_id')['time_ms'].min()
first_seen_sec = (first_seen // 1000).astype(int)
intensity = first_seen_sec.value_counts().sort_index()
for sec, count in intensity.items():
    print(f"сек {sec}: {count} машин")
print(f"\nСредняя: {intensity.mean():.1f} машин/сек")
print(f"Макс: {intensity.max()} машин/сек")

# Сохраняем интенсивность
intensity.to_csv('intensity.csv', header=['count'])
print("\nintensity.csv сохранен")

# СТАТИСТИКА ПО УВЕРЕННОСТИ
print("\nУВЕРЕННОСТЬ ДЕТЕКЦИИ:")
for t in df['class'].unique():
    type_df = df[df['class'] == t]
    print(f"{t}: средняя {type_df['confidence'].mean():.2f}, "
          f"мин {type_df['confidence'].min():.2f}, "
          f"макс {type_df['confidence'].max():.2f}")

# НОВОЕ: СКОРОСТЬ (константа для демонстрации)
print("\nСКОРОСТЬ ДВИЖЕНИЯ (оценка):")
# Для демонстрации используем константные значения
speed_by_type = {
    'car': 80,
    'truck': 60,
    'bus': 60,
    'motorcycle': 50
}

for t in df['class'].unique():
    if t in speed_by_type:
        print(f"{t}: {speed_by_type[t]} км/ч")

# Средняя скорость по всем
avg_speed = np.mean([speed_by_type.get(t, 50) for t in df['class']])
print(f"\nСредняя скорость потока: {avg_speed:.0f} км/ч")

# Сохраняем скорость в отдельный файл
speed_data = []
for t in df['class'].unique():
    if t in speed_by_type:
        speed_data.append({'class': t, 'speed_kmh': speed_by_type[t]})

pd.DataFrame(speed_data).to_csv('speed_estimates.csv', index=False)
print("\nspeed_estimates.csv сохранен")

Сохранено 0 записей


KeyError: 'track_id'